# Red neuronal desde cero con NumPy — MNIST

Implementamos y entrenamos una red neuronal **completamente a mano**: capa densa, activaciones, función de pérdida y *backpropagation* escritos desde cero con `numpy`, y visualizamos con `matplotlib`. Sin frameworks de deep learning.

**Objetivos**
- Entender qué hace una capa densa por dentro: inicializar los pesos, multiplicar matrices y sumas el bias.
- Implementar y visualizar las activaciones ReLU y Softmax.
- Escribir la función de pérdida *cross entropy* y ver cómo baja durante el entrenamiento.
- Derivar a mano los gradientes, capa por capa, aplicando la regla de la cadena.
- Evaluar el modelo y analizar en qué se equivoca.

**Estructura**
1. Carga y exploración del dataset MNIST.
2. Implementación: capa densa, activaciones y funciones de pérdida desde cero.
3. Entrenamiento: *backpropagation* manual y descenso por gradiente.
4. Evaluación: curvas, métricas y análisis de errores.
5. Conclusiones.


> **Cómo ejecutarlo:** recreá el entorno una vez con `uv sync` y elegí el kernel del venv
> (`.venv`). El dataset se descarga solo la primera vez y queda cacheado en `data/mnist.npz`.

## 1. Carga y exploración

MNIST es un dataset de dígitos escritos a mano: 70000 imágenes de 28x28 píxeles en escala de grises (60.000 de entrenamiento y 10.000 de test), cada una etiquetada con su dígito (0-9).

La red va a recibir cada imagen **aplanada**: la matriz de 28x28 se convierte en un vector de 784 valores (se hace más abajo). Primero, los datos.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests

# Semilla global: fija el "azar" para que todas las corridas den lo mismo.
SEED = 42
np.random.seed(SEED)

# Hiperparámetros del entrenamiento, todos en un solo lugar para poder experimentar.
HIDDEN_SIZE = 128       # neuronas de única capa oculta.
BATCH_SIZE = 64         # muestras por actualización de pesos (mini-batch).
LEARNING_RATE = 0.1     # tamaño del paso del descenso por gradiente.
EPOCHS = 10             # pasadas completas sobre el conjunto del entrenamiento.
VAL_SPLIT = 0.1         # fracción del train reservada para validación.

El dataset se descarga con ``requests`` la primera vez y queda guardado en data/mnist.npz. En la próximas ejecuciones se lee directo del disco, sin volver a descargarlo.

In [ ]:
# El notebook vive en notebooks/, así que los datos van a data en la raíz del proyecto.
DATA_DIR = Path("../data")
DATA_FILE = DATA_DIR / "mnist.npz"
URL_MNIST = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"

def descargar_datos(url: str, destino: Path) -> None:
    """Descarga el dataset si todavía no está cacheado en disco."""
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"Dataset ya disponible en {destino} ({destino.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"Descargando {url} ...")
    respuesta = requests.get(url, timeout=120)
    respuesta.raise_for_status()
    destino.write_bytes(respuesta.content)
    print(f"Descarga completa: {destino} ({destino.stat().st_size / 1e6:.1f} MB)")

descargar_datos(URL_MNIST, DATA_FILE)

In [ ]:
datos = np.load(DATA_FILE)
print("Contenido del archivo", sorted(datos.keys()))

x_train_raw = datos["x_train"]  # imágenes de entrenamiento (60000, 28, 28)
y_train = datos["y_train"]      # etiquetas: (60000,)
x_test_raw = datos["x_test"]    # imágenes de test: (10000, 28, 28)
y_test = datos["y_test"]        # etiquetas: (10000,)

print(f"Entrenamiento: {x_train_raw.shape} -> etiquetas {y_train.shape}")
print(f"Test:          {x_test_raw.shape} -> etiquetas {y_test.shape}")
print(f"Valores de píxel: [{x_train_raw.min()}], {x_train_raw.max()}] en uint8")
print(f"Clases posibles: {np.unique(y_train)}")

Veamos algunas imágenes para entender con qué trabaja la red:

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10,5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i], cmap="gray")
    ax.set_title(f"etiqueta: {y_train[i]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Primeras 10 imágenes del set de entrenamiento", fontsize=12)
plt.tight_layout()

Antes de entrenar hacemos tres chequeos y preparativos:

1. **Balance de clases**: que ningún dígito tenga muchos menos ejemplos que otro.
2. **Aplanar**: cada imagen 28×28 pasa a ser un vector de 784 valores, que es lo que espera la capa densa.
3. **Normalizar**: los píxeles van de 0 a 255; los llevamos al rango [0, 1] dividiendo por 255. Con entradas chicas y parejas, el descenso por gradiente converge mucho mejor.

De paso, separamos un 10 % del entrenamiento como **validación**: ejemplos que el modelo nunca
usa para aprender y que nos dicen si está generalizando.

In [ ]:
# Todos los dígitos deberian tener una cantidad similar de ejemplos (~6000 cada uno).
for digito, cantidad in enumerate(np.bincount(y_train, minlength=10)):
    print(f"Dígito {digito}: {cantidad} ejemplos")

In [ ]:
# Aplanar: (N, 28, 28) -> (N, 784) y normalizar a [0, 1].
x_train = x_train_raw.reshape(len(x_train_raw), -1) / 255
x_test = x_test_raw.reshape(len(x_test_raw), -1) / 255

# Partición train/validación. Mezclamos con la semilla fija para que sea reproducible.
indices = np.random.permutation(len(x_train))
n_val = int(len(x_train) * VAL_SPLIT)
indices_val, indices_train = indices[:n_val], indices[n_val:]

x_val, y_val = x_train[indices_val], y_train[indices_val]
x_train, y_train = x_train[indices_train], y_train[indices_train]

print(f"Train:      {x_train.shape}   ->  (784 = 28x28 píxeles aplanados)")
print(f"Validación: {x_val.shape}")
print(f"Test:       {x_test.shape}")